# Hydrogen Data And Scenario Audit

This notebook audits the scenario file used by the hydrogen test case before optimisation runs.

Use this notebook to:
- inspect which scenario artifact is loaded from `scenario_catalog.yaml`
- verify required columns and timestamp fields
- run probability and integrity checks
- inspect example scenario spread versus realised prices

## Required Scenario Columns

The loader maps source columns into this contract:
- `forecast_origin_utc`
- `delivery_start_utc`
- `delivery_start_local`
- `delivery_day`
- `lead_day`
- `model_id`
- `scenario_id`
- `scenario_probability`
- `point_forecast_eur_per_mwh`
- `scenario_price_eur_per_mwh`
- `actual_price_eur_per_mwh`
- `granularity`
- `scenario_generation_run_id`

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
while repo_root.name != "Thesis" and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "scripts" / "Data" / "03_Hydrogen_Test_Case"))

from hydrogen.plant_parameters import load_hydrogen_config
from hydrogen.scenario_loader import load_all_scenarios
from hydrogen.validation_checks import validate_scenario_table

config = load_hydrogen_config(repo_root / "scripts" / "Data" / "03_Hydrogen_Test_Case" / "configs" / "base_hydrogen.yaml")
scenarios, specs, warnings = load_all_scenarios(config)
checks = validate_scenario_table(scenarios)
scenarios.head()

## How To Interpret Probability-Sum Checks

The probability-sum check verifies that probabilities sum to 1 per `(forecast_origin_utc, model_id, lead_day)` block.

- Failing this check means the stochastic objective is economically inconsistent.
- Fix the scenario artifact or mapping before running optimisation.

In [ ]:
import pandas as pd
pd.DataFrame(checks)

## p95 Undercoverage Interpretation

p95 undercoverage means realised prices are outside the model's upper scenario band more often than expected.

This is not automatically a bug. It can indicate poor tail calibration, distribution shift, or insufficient stress scenarios.
Treat this as a calibration warning signal, not a proof of implementation failure.

In [ ]:
import matplotlib.pyplot as plt

example = scenarios[scenarios["model_id"] == scenarios["model_id"].iloc[0]].copy()
example = example[example["delivery_day"] == example["delivery_day"].iloc[0]].copy()
q = example.groupby("delivery_start_utc").agg(
    actual=("actual_price_eur_per_mwh", "first"),
    point=("point_forecast_eur_per_mwh", "first"),
    p05=("scenario_price_eur_per_mwh", lambda s: s.quantile(0.05)),
    p95=("scenario_price_eur_per_mwh", lambda s: s.quantile(0.95)),
).reset_index()

plt.figure(figsize=(12, 4))
plt.fill_between(q["delivery_start_utc"], q["p05"], q["p95"], alpha=0.2, label="Scenario p05-p95")
plt.plot(q["delivery_start_utc"], q["point"], label="Point forecast")
plt.plot(q["delivery_start_utc"], q["actual"], label="Actual")
plt.legend()
plt.title("Example Scenario Spread")
plt.show()